# Project SQL - BigQuery

## Pick a dataset that interests you (or multiple data sets)

Use the Open Data Sets available from Google BigQuery. You can use your own Google account or Kaggle.



## Come up with questions about your data
* What sort of information is in this dataset?
* How many records are there?
* Have the number of bitcoin transactions increased year over year?
* Does New Mexico get more or less rain now than 20 years ago?
* How many different countries (states, counties, cities, etc) have records in this data set?




## Use SQL queries to pull specific information

Like with the Chinook data base, do NOT pull all the data and then filter using DataFrame methods etc.

Write your question and then answer it using the SQL statements below.  Be sure to use guardrails ( e.g. dryrun ).



##GOAL

 which is the country with the most purchases of music? and what are the complaints of those countries to better optimize top country satisfaction? will be looking joining and discecting invoices,  invoice_items, customers

In [2]:
# which is the country with the most purchases of music? and what are the complaints of those countries to better optimize top country satisfaction? will be looking joining and discecting invoices and invoice_items

In [3]:
# will possibly also need to add on another thing

In [4]:
import sqlite3 as db
import pandas as pd


In [5]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3


In [6]:
!sqlite3 --version

3.37.2 2022-01-06 13:25:41 872ba256cbf61d9290b571c0e6d82a20c224ca3ad82971edc46b29818d5dalt1


In [7]:
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip


Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [8]:
!unzip -u chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [9]:
!ls -la

total 1180
drwxr-xr-x 1 root root   4096 Jul  9 15:39 .
drwxr-xr-x 1 root root   4096 Jul  9 15:36 ..
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Jul  9 15:39 chinook.zip
drwxr-xr-x 4 root root   4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root   4096 Jun  4 13:32 sample_data


In [10]:
query = '''
  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;
'''
print(query)


  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;



In [11]:
db_con = db.connect("chinook.db")
tables = pd.read_sql_query( query , db_con)

tables


,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoices
6,invoice_items
7,media_types
8,playlists
9,playlist_track


In [12]:
table_names_query = "SELECT name FROM sqlite_schema WHERE type='table';"
table_list = pd.read_sql_query(table_names_query, db_con)['name'].tolist()

print("--- Table Row Counts ---")

for table_name in table_list:
    count_query = f"SELECT COUNT(*) AS total_rows FROM [{table_name}]"
    count_df = pd.read_sql_query(count_query, db_con)
    row_count = count_df.iloc[0]['total_rows']

    print(f"Table: {table_name:<20} | Rows: {row_count}")

--- Table Row Counts ---
Table: albums               | Rows: 347
Table: sqlite_sequence      | Rows: 10
Table: artists              | Rows: 275
Table: customers            | Rows: 59
Table: employees            | Rows: 8
Table: genres               | Rows: 25
Table: invoices             | Rows: 412
Table: invoice_items        | Rows: 2240
Table: media_types          | Rows: 5
Table: playlists            | Rows: 18
Table: playlist_track       | Rows: 8715
Table: tracks               | Rows: 3503
Table: sqlite_stat1         | Rows: 15


### Basic Queries


#### SELECT (with * and with column names)


In [13]:
query =( '''
SELECT
    i.BillingCountry,
    g.Name AS GenreName,
    SUM(ii.Quantity) AS TracksSold
FROM genres g
INNER JOIN tracks t ON g.GenreId = t.GenreId
INNER JOIN invoice_items ii ON t.TrackId = ii.TrackId
INNER JOIN invoices i ON ii.InvoiceId = i.InvoiceId
GROUP BY i.BillingCountry, g.Name
ORDER BY TracksSold DESC
LIMIT(10)
''')
artist = pd.read_sql_query(query, db_con)
artist

,BillingCountry,GenreName,TracksSold
0,USA,Rock,157
1,Canada,Rock,107
2,USA,Latin,91
3,Brazil,Rock,81
4,France,Rock,65
5,USA,Metal,64
6,Germany,Rock,62
7,Canada,Latin,60
8,Brazil,Latin,53
9,USA,Alternative & Punk,50


In [14]:
query = '''
  SELECT *
  FROM genres


'''

artist = pd.read_sql_query(query, db_con)
artist

,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll
5,6,Blues
6,7,Latin
7,8,Reggae
8,9,Pop
9,10,Soundtrack


#### WHERE


In [15]:
query = '''
  SELECT *
  FROM invoice_items
  WHERE UnitPrice >= 0.99

'''

artist = pd.read_sql_query(query, db_con)
artist

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1
2,3,2,6,0.99,1
3,4,2,8,0.99,1
4,5,2,10,0.99,1
...,...,...,...,...,...
2235,2236,411,3136,0.99,1
2236,2237,411,3145,0.99,1
2237,2238,411,3154,0.99,1
2238,2239,411,3163,0.99,1


In [16]:
# there are 2 different price rannges 0.99 and 1.99 verified by adjusting the WHERE
# what determines the price of 1.99 and 0.99

In [17]:
query = '''
  SELECT *
  FROM invoices

'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2009-01-02 00:00:00,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2009-01-03 00:00:00,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2009-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2009-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86
...,...,...,...,...,...,...,...,...,...
407,408,25,2013-12-05 00:00:00,319 N. Frances Street,Madison,WI,USA,53703,3.96
408,409,29,2013-12-06 00:00:00,796 Dundas Street West,Toronto,ON,Canada,M6J 1V1,5.94
409,410,35,2013-12-09 00:00:00,"Rua dos Campeões Europeus de Viena, 4350",Porto,None,Portugal,None,8.91
410,411,44,2013-12-14 00:00:00,Porthaninkatu 9,Helsinki,None,Finland,00530,13.86


In [18]:
query = '''
  SELECT *
  FROM invoices
  WHERE BillingCountry = 'Germany'
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,6,37,2009-01-19 00:00:00,Berger Straße 10,Frankfurt,None,Germany,60316,0.99
2,7,38,2009-02-01 00:00:00,Barbarossastraße 19,Berlin,None,Germany,10779,1.98
3,12,2,2009-02-11 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,13.86
4,29,36,2009-05-05 00:00:00,Tauentzienstraße 8,Berlin,None,Germany,10789,1.98
5,30,38,2009-05-06 00:00:00,Barbarossastraße 19,Berlin,None,Germany,10779,3.96
6,40,36,2009-06-15 00:00:00,Tauentzienstraße 8,Berlin,None,Germany,10789,13.86
7,52,38,2009-08-08 00:00:00,Barbarossastraße 19,Berlin,None,Germany,10779,5.94
8,67,2,2009-10-12 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,8.91
9,95,36,2010-02-13 00:00:00,Tauentzienstraße 8,Berlin,None,Germany,10789,8.91


In [19]:
query = '''
  SELECT
    BillingCountry,
    SUM(Total) AS TotalSales
FROM invoices
GROUP BY BillingCountry
ORDER BY TotalSales DESC;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry,TotalSales
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48
5,United Kingdom,112.86
6,Czech Republic,90.24
7,Portugal,77.24
8,India,75.26
9,Chile,46.62


In [20]:
# how many people purchased music in each individual country? the total of people is 59 customers.

In [21]:
query = '''

SELECT
    BillingCountry,
    COUNT(DISTINCT CustomerId) AS UniqueCustomers
FROM invoices
GROUP BY BillingCountry
ORDER BY UniqueCustomers DESC;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry,UniqueCustomers
0,USA,13
1,Canada,8
2,France,5
3,Brazil,5
4,Germany,4
5,United Kingdom,3
6,Portugal,2
7,India,2
8,Czech Republic,2
9,Sweden,1


In [22]:
# would like to independently look at each of the top country with the most purchases to see what kind of music they are buying
# I will be joining USA invoices with
query = '''
 SELECT
    BillingCountry,
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total
FROM invoices
WHERE BillingCountry = 'USA'
ORDER BY Total DESC;
'''

invoices_USA = pd.read_sql_query(query, db_con)
invoices_USA

,BillingCountry,InvoiceId,CustomerId,InvoiceDate,Total
0,USA,299,26,2012-08-05 00:00:00,23.86
1,USA,201,25,2011-05-29 00:00:00,18.86
2,USA,103,24,2010-03-21 00:00:00,15.86
3,USA,5,23,2009-01-11 00:00:00,13.86
4,USA,26,19,2009-04-14 00:00:00,13.86
...,...,...,...,...,...
86,USA,265,27,2012-03-11 00:00:00,0.99
87,USA,286,23,2012-06-12 00:00:00,0.99
88,USA,363,28,2013-05-19 00:00:00,0.99
89,USA,384,24,2013-08-20 00:00:00,0.99


In [23]:
query = '''
WITH USA_Invoices AS (

    SELECT InvoiceId, BillingCountry
    FROM invoices
    WHERE BillingCountry = 'USA'
)

SELECT
    u.BillingCountry,
    g.Name AS GenreName,
    SUM(ii.Quantity) AS TracksSold
FROM USA_Invoices u
INNER JOIN invoice_items ii ON u.InvoiceId = ii.InvoiceId
INNER JOIN tracks t         ON ii.TrackId = t.TrackId
INNER JOIN genres g         ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY TracksSold DESC;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry,GenreName,TracksSold
0,USA,Rock,157
1,USA,Latin,91
2,USA,Metal,64
3,USA,Alternative & Punk,50
4,USA,Jazz,22
5,USA,Blues,15
6,USA,TV Shows,14
7,USA,R&B/Soul,12
8,USA,Comedy,8
9,USA,Classical,8


In [24]:
#looking at the nununique values of those who purchased in USA, to see each individuals transaction history to see the genre that they

In [25]:
query = '''
SELECT
    COUNT(DISTINCT InvoiceId) AS UniqueUSAInvoiceId
FROM invoices
WHERE BillingCountry = 'USA';
'''
invoices_s = pd.read_sql_query(query, db_con)
invoices_s
#looks like each purchace has a unique invoice ID next step is to join it with invoice items and then to tracks and then possibly genres

,UniqueUSAInvoiceId
0,91


##join

In [26]:
query_join = '''
  SELECT *
  FROM invoices


'''

invoices_s = pd.read_sql_query(query_join, db_con)
invoices_s

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2009-01-02 00:00:00,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2009-01-03 00:00:00,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2009-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2009-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86
...,...,...,...,...,...,...,...,...,...
407,408,25,2013-12-05 00:00:00,319 N. Frances Street,Madison,WI,USA,53703,3.96
408,409,29,2013-12-06 00:00:00,796 Dundas Street West,Toronto,ON,Canada,M6J 1V1,5.94
409,410,35,2013-12-09 00:00:00,"Rua dos Campeões Europeus de Viena, 4350",Porto,None,Portugal,None,8.91
410,411,44,2013-12-14 00:00:00,Porthaninkatu 9,Helsinki,None,Finland,00530,13.86


In [27]:
# want see which columns compares with the table above to join them
query_I = '''
  SELECT *
  FROM invoice_items

'''

invoices_s = pd.read_sql_query(query_I, db_con)
invoices_s

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1
2,3,2,6,0.99,1
3,4,2,8,0.99,1
4,5,2,10,0.99,1
...,...,...,...,...,...
2235,2236,411,3136,0.99,1
2236,2237,411,3145,0.99,1
2237,2238,411,3154,0.99,1
2238,2239,411,3163,0.99,1


In [28]:
# seeing the nuite prices and there are 2 set prices one for .99 and another for 1.99
query = '''
  SELECT *
  FROM tracks
  WHERE UnitPrice >= 0.99

'''

artist = pd.read_sql_query(query, db_con)
artist

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99
...,...,...,...,...,...,...,...,...,...
3498,3499,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,343,2,24,None,286741,4718950,0.99
3499,3500,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",344,2,24,Franz Schubert,139200,2283131,0.99
3500,3501,"L'orfeo, Act 3, Sinfonia (Orchestra)",345,2,24,Claudio Monteverdi,66639,1189062,0.99
3501,3502,"Quintet for Horn, Violin, 2 Violas, and Cello ...",346,2,24,Wolfgang Amadeus Mozart,221331,3665114,0.99


In [29]:
query = '''
 SELECT
    BillingCountry,
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total
FROM invoices
WHERE BillingCountry = 'Canada'
ORDER BY Total DESC;
'''

invoices_C = pd.read_sql_query(query, db_con)
invoices_C

,BillingCountry,InvoiceId,CustomerId,InvoiceDate,Total
0,Canada,47,15,2009-07-16 00:00:00,13.86
1,Canada,61,32,2009-09-16 00:00:00,13.86
2,Canada,110,3,2010-04-21 00:00:00,13.86
3,Canada,159,33,2010-11-24 00:00:00,13.86
4,Canada,180,29,2011-02-25 00:00:00,13.86
5,Canada,278,30,2012-05-04 00:00:00,13.86
6,Canada,362,14,2013-05-11 00:00:00,13.86
7,Canada,376,31,2013-07-12 00:00:00,13.86
8,Canada,102,15,2010-03-16 00:00:00,9.91
9,Canada,4,14,2009-01-06 00:00:00,8.91


In [30]:
query = '''
WITH Canada_Invoices AS (
    -- CTE: Filter down to just USA invoices first
    SELECT InvoiceId, BillingCountry
    FROM invoices
    WHERE BillingCountry = 'Canada'
)

SELECT
    u.BillingCountry,
    g.Name AS GenreName,
    SUM(ii.Quantity) AS TracksSold
FROM Canada_Invoices u
INNER JOIN invoice_items ii ON u.InvoiceId = ii.InvoiceId
INNER JOIN tracks t         ON ii.TrackId = t.TrackId
INNER JOIN genres g         ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY TracksSold DESC;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry,GenreName,TracksSold
0,Canada,Rock,107
1,Canada,Latin,60
2,Canada,Metal,40
3,Canada,Alternative & Punk,36
4,Canada,Jazz,13
5,Canada,Reggae,7
6,Canada,Bossa Nova,7
7,Canada,World,6
8,Canada,R&B/Soul,5
9,Canada,Hip Hop/Rap,5


In [31]:
query = '''
 SELECT
    BillingCountry,
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total
FROM invoices
WHERE BillingCountry = 'France'
ORDER BY Total DESC;
'''

invoices_F = pd.read_sql_query(query, db_con)
invoices_F

,BillingCountry,InvoiceId,CustomerId,InvoiceDate,Total
0,France,313,43,2012-10-06 00:00:00,16.86
1,France,19,40,2009-03-14 00:00:00,13.86
2,France,117,41,2010-05-22 00:00:00,13.86
3,France,215,42,2011-07-30 00:00:00,13.86
4,France,334,39,2013-01-07 00:00:00,13.86
5,France,74,40,2009-11-12 00:00:00,8.91
6,France,172,41,2011-01-20 00:00:00,8.91
7,France,270,42,2012-03-29 00:00:00,8.91
8,France,368,43,2013-06-06 00:00:00,8.91
9,France,389,39,2013-09-07 00:00:00,8.91


In [32]:
query = '''
WITH France_Invoices AS (

    SELECT InvoiceId, BillingCountry
    FROM invoices
    WHERE BillingCountry = 'France'
)

SELECT
    u.BillingCountry,
    g.Name AS GenreName,
    SUM(ii.Quantity) AS TracksSold
FROM France_Invoices u
INNER JOIN invoice_items ii ON u.InvoiceId = ii.InvoiceId
INNER JOIN tracks t         ON ii.TrackId = t.TrackId
INNER JOIN genres g         ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY TracksSold DESC;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry,GenreName,TracksSold
0,France,Rock,65
1,France,Alternative & Punk,31
2,France,Latin,26
3,France,Metal,20
4,France,Jazz,11
5,France,Classical,10
6,France,Soundtrack,5
7,France,Drama,4
8,France,Alternative,4
9,France,Sci Fi & Fantasy,2


In [33]:
query = '''
 SELECT
    BillingCountry,
    InvoiceId,
    CustomerId,
    InvoiceDate,
    Total
FROM invoices
WHERE BillingCountry = 'Brazil'
ORDER BY Total DESC;
'''

invoices_B = pd.read_sql_query(query, db_con)
invoices_B

,BillingCountry,InvoiceId,CustomerId,InvoiceDate,Total
0,Brazil,68,11,2009-10-17 00:00:00,13.86
1,Brazil,166,12,2010-12-25 00:00:00,13.86
2,Brazil,264,13,2012-03-03 00:00:00,13.86
3,Brazil,327,1,2012-12-07 00:00:00,13.86
4,Brazil,383,10,2013-08-12 00:00:00,13.86
5,Brazil,25,10,2009-04-09 00:00:00,8.91
6,Brazil,123,11,2010-06-17 00:00:00,8.91
7,Brazil,221,12,2011-08-25 00:00:00,8.91
8,Brazil,319,13,2012-11-01 00:00:00,8.91
9,Brazil,382,1,2013-08-07 00:00:00,8.91


In [34]:
query = '''
WITH Brazil_Invoices AS (
    -- CTE: Filter down to just USA invoices first
    SELECT InvoiceId, BillingCountry
    FROM invoices
    WHERE BillingCountry = 'Brazil'
)
-- Main Query: Join the clean CTE to the track and genre tables
SELECT
    u.BillingCountry,
    g.Name AS GenreName,
    SUM(ii.Quantity) AS TracksSold
FROM Brazil_Invoices u
INNER JOIN invoice_items ii ON u.InvoiceId = ii.InvoiceId
INNER JOIN tracks t         ON ii.TrackId = t.TrackId
INNER JOIN genres g         ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY TracksSold DESC;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry,GenreName,TracksSold
0,Brazil,Rock,81
1,Brazil,Latin,53
2,Brazil,Metal,15
3,Brazil,Alternative & Punk,7
4,Brazil,Reggae,6
5,Brazil,Classical,6
6,Brazil,Blues,6
7,Brazil,Soundtrack,4
8,Brazil,R&B/Soul,3
9,Brazil,Pop,3


In [35]:
# trying to know how many countries have purchased/billed
query = '''
  SELECT COUNT(DISTINCT Billingcountry) AS unique_count
FROM invoices;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,unique_count
0,24


In [36]:
query = '''
  SELECT DISTINCT Billingcountry
FROM invoices;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

,BillingCountry
0,Germany
1,Norway
2,Belgium
3,Canada
4,USA
5,France
6,Ireland
7,United Kingdom
8,Australia
9,Chile


In [37]:
# I would like to add a column with the totals to see what country purchases more

In [ ]:
# im gonna find out if there are them same amount of customers in both invoices and invoice_items
query = '''
  SELECT COUNT(DISTINCT CustomerId) AS how_md_customers
FROM invoices;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

In [ ]:
#seems like to me that the invoices relates to customers. the unique characters of
query = '''
  SELECT COUNT(DISTINCT CustomerId) AS how_md_customers
FROM customers;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

In [ ]:
query = '''
  SELECT COUNT(DISTINCT InvoiceId) AS how_md_customers
FROM invoice_items;
'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

In [ ]:
# they dont match but the amount of unique customers in invoice_items equals the

In [ ]:
query = '''
  SELECT *
  FROM tracks

'''

invoices_s = pd.read_sql_query(query, db_con)
invoices_s

In [ ]:
query = '''
  SELECT *
  FROM tracks
  WHERE UnitPrice >= 0.99

'''

artist = pd.read_sql_query(query, db_con)
artist

In [ ]:
# I will be joining tracks

In [ ]:
query = '''
  SELECT *
  FROM genres


'''

artist = pd.read_sql_query(query, db_con)
artist

## Make some plots

Make some cool plots to go with your data. Write SQL queries to get ONLY the information you need for each plot. (Don't pull ALL the data and then just plot a few columns.)



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
query = '''
SELECT
    BillingCountry,
    SUM(Total) AS TotalSales
FROM invoices
GROUP BY BillingCountry
ORDER BY TotalSales DESC;
'''
invoices_s = pd.read_sql_query(query, db_con)

In [ ]:

sns.set_theme(style="whitegrid")


plt.figure(figsize=(12, 7))


sns.barplot(data=invoices_s, x='BillingCountry', y='TotalSales', hue='BillingCountry', palette='viridis', legend=False)


plt.title('Total Music Sales by Country', fontsize=14, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Total Sales ($)', fontsize=12)


plt.xticks(rotation=90)


plt.show()

##Parquet


In [43]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  for table in tables ['name']:
    pirnt(table)
    query = f'''
      select* from {table};
      '''
    df = pd.read_sql_query (query, db_con)
    display(df.shape)
    output_path = f'./chinook/{table}.parquet'
    print(output_path)
    df.to_parquet(output_path, path = True)

NameError: name 'contextlib' is not defined

In [ ]:
# if you want to save memory you can individually change the in64 to int32 or int16 or int 8. check it out and see how much integers there are and get rid of excess space
# same thinf with float64, and object

# not in csv there are not such thing as data types so all the saving up of space is not worth it
# parquet has the advantage because memory can be saved

In [ ]:
# duckdb must be imported. duckdb

conn - duckdb
conn. execute ("LOAD sqlite;")
conn.excecute(ATTACH chinook.db AS chinook (type ))

In [ ]:
query = r''

##pushing hugging face

In [ ]:
hf_url = ''

#print to see that it worked
hf_url